
# **Creating Delta table**

### **using SQL API**

In [0]:
%sql
CREATE TABLE deltalake.default.deltalake1table(
  id INT,
  salary INT
)

### ***Using Delta API***

In [0]:
from delta.tables import DeltaTable

In [0]:
DeltaTable.createIfNotExists(spark)\
    .tableName("deltalake.default.firstDeltaAPI")\
    .addColumn("id","INT")\
    .addColumn("salary","INT")\
    .execute()

# **GENERATED COLUMNS**

### **Identity Generator**

In [0]:
from delta.tables import IdentityGenerator
from pyspark.sql.types import LongType, IntegerType, StringType

In [0]:
# Identity column with default settings (starts at 1, increments by 1)

DeltaTable.createIfNotExists(spark)\
  .tableName("deltalake.default.identityColumn")\
  .addColumn("id", dataType=LongType(), generatedAlwaysAs=IdentityGenerator())\
  .addColumn("name", dataType=StringType())\
  .addColumn("salary", dataType=IntegerType())\
  .execute()

# Alternative: Identity column with custom start and increment
# DeltaTable.createIfNotExists(spark)\
#   .tableName("deltalake.default.customIdentity")\
#   .addColumn("id", dataType=LongType(), generatedAlwaysAs=IdentityGenerator(start=100, step=10))\
#   .addColumn("name", "STRING")\
#   .execute()

In [0]:
# Insert data - id column is auto-generated
spark.sql("""
  INSERT INTO deltalake.default.identityColumn (name, salary)
  VALUES 
    ('Alice', 75000),
    ('Bob', 80000),
    ('Charlie', 90000)
""")

# View the data - notice the auto-generated id values
display(spark.table("deltalake.default.identityColumn"))

### **Computed Columns**

In [0]:
# Drop existing table to update the computed column formula
spark.sql("DROP TABLE IF EXISTS deltalake.default.salary")

# Recreate with 10% calculation
DeltaTable.create(spark)\
    .tableName("deltalake.default.salary")\
    .addColumn("name",dataType=StringType())\
    .addColumn("salary", dataType=IntegerType())\
    .addColumn("salaryAfterTax",dataType=LongType(),generatedAlwaysAs="CAST(salary*0.1 AS BIGINT)")\
    .execute()

In [0]:
%sql
insert into deltalake.default.salary(name,salary)
values ("harish", 45000), ("babu",32434);

In [0]:
%sql
select * from deltalake.default.salary

In [0]:
%sql
insert into deltalake.default.deltalake1table (id, salary)
values (1,24453),(2,34353),(3,45445)